# RLS Online - Identificação do Aeropêndulo
Este notebook realiza a estimação online utilizando Recursive Least Squares (RLS), puxando os dados diretamente do repositório online.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Carregamento Online dos Dados
Os dados da RODADA-7 são extraídos diretamente do github raw content.

In [ ]:
def carregar_experimento(url, decimacao=1, start_idx=0, end_idx=None):
    df = pd.read_csv(url)
    df_sub = df.iloc[::decimacao].copy().reset_index(drop=True)
    
    if 'u_pct' in df_sub.columns:
        u_raw = df_sub['u_pct'].values
    elif 'motor_percent' in df_sub.columns:
        u_raw = df_sub['motor_percent'].values
    else:
        raise ValueError("Coluna de controle não encontrada!")
        
    y_raw = df_sub['angulo_deg'].values
    
    if end_idx is None:
        end_idx_real = len(u_raw)
    elif end_idx < 0:
        end_idx_real = len(u_raw) + end_idx
    else:
        end_idx_real = end_idx
        
    u = u_raw[start_idx:end_idx_real]
    y = y_raw[start_idx:end_idx_real]
    return u, y

BASE2 = (
    "https://raw.githubusercontent.com/FelipeEduardoMarcondes/"
    "SYSTEM-IDENTIFICATION-AERO/main/experimentos/"
)

CUT_TRA_START = 1500
CUT_TRA_END = -1400

# Carregamento online
url_treino = BASE2 + "RODADA-7/chirp-60-amp50_0908_23-09.csv"
url_teste  = BASE2 + "RODADA-7/aprbs-45-1_0904_19-54.csv"

u_TRA, y_TRA = carregar_experimento(url_treino, start_idx=CUT_TRA_START, end_idx=CUT_TRA_END)
u_TEST, y_TEST = carregar_experimento(url_teste, start_idx=CUT_TRA_START, end_idx=CUT_TRA_END)

print(f"Amostras de Treinamento: {len(u_TRA)}")
print(f"Amostras de Teste: {len(u_TEST)}")

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(u_TRA, color='green')
axes[0].set_title('Entrada (Treinamento)')
axes[1].plot(y_TRA, color='blue')
axes[1].set_title('Saída (Treinamento)')
plt.tight_layout()
plt.show()

## 2. Implementação do RLS e BLS (Baseline)

In [ ]:
na = 2
nb = 2
p = max(na, nb)
N = len(u_TRA)
num_params = na + nb

# --- BATCH LEAST SQUARES (BLS) ---
Phi_bls = np.zeros((N - p, num_params))
y_target = y_TRA[p:]

for i in range(1, na + 1):
    Phi_bls[:, i - 1] = -y_TRA[p - i: N - i]
for i in range(1, nb + 1):
    Phi_bls[:, na + i - 1] = u_TRA[p - i: N - i]

theta_bls, _, _, _ = np.linalg.lstsq(Phi_bls, y_target, rcond=None)
print(f"Parâmetros BLS finais: {theta_bls}")

# --- RECURSIVE LEAST SQUARES (RLS) ---
theta_rls = np.zeros((num_params, 1))
P = np.eye(num_params) * 1000.0  # Alta incerteza inicial
lam = 1.0  # Fator de esquecimento (1.0 = sem esquecimento)

theta_history = np.zeros((N, num_params))
y_est_rls = np.zeros(N)

for k in range(p, N):
    # Vetor de regressores phi(k)
    phi_k = np.zeros((num_params, 1))
    for i in range(1, na + 1):
        phi_k[i - 1, 0] = -y_TRA[k - i]
    for i in range(1, nb + 1):
        phi_k[na + i - 1, 0] = u_TRA[k - i]
        
    # Ganho de Kalman
    P_phi = P @ phi_k
    K = P_phi / (lam + phi_k.T @ P_phi)
    
    # Previsão um passo à frente
    y_hat = (phi_k.T @ theta_rls).item()
    y_est_rls[k] = y_hat
    
    # Erro de predição
    e = y_TRA[k] - y_hat
    
    # Atualização dos parâmetros
    theta_rls = theta_rls + K * e
    
    # Atualização da matriz de covariância
    P = (P - K @ phi_k.T @ P) / lam
    
    # Histórico
    theta_history[k, :] = theta_rls.flatten()

print(f"Parâmetros RLS finais: {theta_rls.flatten()}")

## 3. Visualização dos Parâmetros

In [ ]:
fig, axes = plt.subplots(num_params, 1, figsize=(10, 8), sharex=True)
fig.suptitle('Evolução dos Parâmetros Estimados (RLS) vs Baseline (BLS)', fontsize=14)

t = np.arange(p, N)
labels = [f'a{i}' for i in range(1, na+1)] + [f'b{i}' for i in range(1, nb+1)]

for i in range(num_params):
    axes[i].plot(t, theta_history[p:, i], label=f'{labels[i]} (RLS)', color='blue', lw=2)
    axes[i].axhline(theta_bls[i], color='red', linestyle='--', label=f'{labels[i]} (BLS)')
    axes[i].set_ylabel(f'Parâmetro {labels[i]}')
    axes[i].grid(True)
    axes[i].legend(loc='upper right')

axes[-1].set_xlabel('Amostras (k)')
plt.tight_layout()
plt.show()

## 4. Validação da Predição (Treinamento)

In [ ]:
y_est_bls = np.zeros(N)
y_est_bls[:p] = y_TRA[:p]
y_est_bls[p:] = Phi_bls @ theta_bls

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(t, y_TRA[p:], color='black', lw=2, label='Saída Medida (Real)')
ax.plot(t, y_est_bls[p:], color='red', ls='--', lw=2, alpha=0.8, label='Predição BLS (OSA)')
ax.plot(t, y_est_rls[p:], color='blue', ls='-.', lw=2, alpha=0.8, label='Predição RLS (OSA)')

ax.set_title('Saída Real vs Estimadas (Treinamento)')
ax.set_xlabel('Amostras (k)')
ax.set_ylabel('Ângulo (deg)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()